In [ ]:
%load_ext autoreload
%autoreload 2

import os
from os.path import dirname, abspath
from copy import deepcopy

import omegaconf
from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir

import graphium
from graphium.config._loader import (
    load_accelerator,
    load_datamodule,
    load_architecture,
    load_metrics,
    load_predictor,
)

## Read the config file

In [ ]:
# Set up the working directory to graphium project root
MAIN_DIR = dirname(dirname(abspath(graphium.__file__)))
os.chdir(MAIN_DIR)

# Load the Hydra config (toymix + GCN as a minimal example)
config_dir = os.path.join(MAIN_DIR, "expts", "hydra-configs")
with initialize_config_dir(version_base=None, config_dir=config_dir):
    cfg = compose(config_name="main", overrides=["model=gcn", "accelerator=cpu"])

cfg = OmegaConf.to_container(cfg, resolve=True)
print("Config loaded. Keys:", list(cfg.keys()))

## Load a dataset

In [ ]:
# Load accelerator config and create the datamodule
cfg, accelerator_type = load_accelerator(cfg)

datamodule = load_datamodule(cfg, accelerator_type)
datamodule.prepare_data()

print(f"Accelerator: {accelerator_type}")
print(f"Datamodule type: {type(datamodule).__name__}")
print(f"Input dims: {datamodule.in_dims}")

In [ ]:
# Build the model architecture from config
model_class, model_kwargs = load_architecture(cfg, in_dims=datamodule.in_dims)

model = model_class(**model_kwargs)
print(f"\nModel class: {model_class.__name__}")
print(model)

In [ ]:
# Load metrics
metrics = load_metrics(cfg)
for task, task_metrics in metrics.items():
    print(f"{task}: {list(task_metrics.keys())}")

In [ ]:
# Build the full predictor (model + optimizer + metrics)
predictor = load_predictor(
    config=cfg,
    model_class=model_class,
    model_kwargs=model_kwargs,
    metrics=metrics,
    task_levels=datamodule.get_task_levels(),
    accelerator_type=accelerator_type,
    featurization=datamodule.featurization,
    task_norms=datamodule.task_norms,
)

print(f"Predictor ready. Tasks: {list(predictor.model.task_heads.keys())}")